# SmallG1OnlineV1a on Modal

This notebook builds a reproducible Modal image from `bsmathers/testing2`, prepares the exact epoch-475 TaurosV1B-compatible policy, assembles all 807 expert teams **without deduplication**, downloads the public replay/team/usage-stat datasets and historical opponent weights, prefills the online FIFO on CPUs, and launches the audited online learner on one NVIDIA GPU with Weights & Biases logging.

The bootstrap function accepts epoch 475 in four forms, in order: three Git-tracked shards under `policy/`, an explicit HTTPS URL, a monolithic GitHub Release asset, or three sharded Release assets. A pre-uploaded Modal copy remains the fallback. `good_teams` can similarly come from a release/URL or a one-time Modal upload.

Operational choices: two v1 Modal Volumes are deliberate. The run's 300,000-file FIFO exceeds the v2 per-directory limit of 262,144, while putting both the extracted offline corpus and FIFO in one v1 Volume risks its 500,000-inode cap. `smallg1onlinev1a-data` holds immutable datasets/bootstrap/config; `smallg1onlinev1a-run` holds the FIFO and checkpoints. A separate CPU prefill avoids paying for a GPU while the FIFO reaches 5,001 files. The learner and ongoing collectors then share one container, so new replay files are immediately visible without cross-container volume reloads.

## One-time local setup

Run these in a terminal on the machine that has `~/testing2`. Do not create either Volume as v2. `HF_TOKEN` is optional for public files, but avoids anonymous Hugging Face throttling.

```bash
python -m pip install -U modal
modal setup
modal volume create smallg1onlinev1a-data
modal volume create smallg1onlinev1a-run
modal secret create smallg1onlinev1a-secrets WANDB_API_KEY="$WANDB_API_KEY" HF_TOKEN="$HF_TOKEN"

# Fallback only when the checkpoint shards and team archive are not on GitHub:
modal volume put -f smallg1onlinev1a-data ~/testing2/policy_epoch_475.pt /bootstrap/policy_epoch_475.pt
modal volume put -f smallg1onlinev1a-data ~/testing2/teams_replay/good_teams /bootstrap/good_teams/

modal volume ls smallg1onlinev1a-data /bootstrap
```

If the three checkpoint shards are committed under `policy/`, omit the checkpoint upload. If the team archive is published with the exact name `good_teams.tar.gz`, `good_teams.tgz`, or `good_teams.zip`, omit the team upload.

## Create the three GitHub-safe checkpoint shards

The checkpoint is about 237 MiB, so the intended unit is **MiB**, not GiB. GitHub blocks ordinary Git objects above 100 MiB. From `~/testing2`, create three numeric, byte-for-byte parts of at most 85 MiB and a checksum manifest:

```bash
mkdir -p policy
split -d -a 3 -b 85m policy_epoch_475.pt policy/policy_epoch_475.pt.part-
shasum -a 256 policy_epoch_475.pt > policy/policy_epoch_475.pt.sha256
ls -lh policy/policy_epoch_475.pt.part-*
git add policy/policy_epoch_475.pt.part-* policy/policy_epoch_475.pt.sha256
git commit -m "Add sharded epoch-475 policy"
git push origin main
```

Expected part sizes for the current 248,408,973-byte file are 85 MiB, 85 MiB, and about 66.9 MiB. Do **not** add the original `.pt` file. The Modal bootstrap concatenates parts in lexical order, verifies the SHA-256 manifest, then applies `export-online`. Use a branch/PR instead of pushing `main` directly if branch protection requires it.

In [ ]:
%pip install -q -U modal requests


In [ ]:
import os
import requests
import modal

REPO = "bsmathers/testing2"
REPO_URL = f"https://github.com/{REPO}.git"
BRANCH = "main"
DATA_VOLUME_NAME = "smallg1onlinev1a-data"
RUN_VOLUME_NAME = "smallg1onlinev1a-run"
SECRET_NAME = "smallg1onlinev1a-secrets"
GPU = "L40S"                 # Change to A100-80GB or H100 for benchmarking.
WEIGHTS_URL = ""              # Optional direct HTTPS URL; exact epoch 475 only.
TEAMS_URL = ""                # Optional .tar.gz/.tgz/.zip containing all 807 files.
COLLECTOR_PROCESSES = 4
LANES_PER_COLLECTOR = 32

api = f"https://api.github.com/repos/{REPO}/commits/{BRANCH}"
response = requests.get(api, timeout=30)
response.raise_for_status()
GIT_SHA = response.json()["sha"]
print("Pinned repository commit:", GIT_SHA)


In [ ]:
image = (
    modal.Image.from_registry(
        "nvidia/cuda:12.8.1-cudnn-devel-ubuntu22.04", add_python="3.11"
    )
    .apt_install("git", "git-lfs", "curl", "ca-certificates", "gnupg", "build-essential")
    .env({
        "MAX_JOBS": "4",
        "PYTHONUNBUFFERED": "1",
        "METAMON_CACHE_DIR": "/data/cache",
        "METAMON_SAVE_DIR": "/run/checkpoints",
        # Makes the local TaurosV1B registry entry resolve the frozen epoch-475 opponent.
        "TAUROSV1B_RUN_NAME": "epoch475_frozen",
        "METAMON_WANDB_PROJECT": "smallg1onlinev1a",
        "WANDB_RUN_ID": "smallg1onlinev1a-modal-v1",
        "WANDB_NAME": "smallg1onlinev1a",
        "WANDB_RESUME": "allow",
        "HF_XET_HIGH_PERFORMANCE": "1",
    })
    .run_commands(
        f"git clone {REPO_URL} /opt/testing2",
        f"cd /opt/testing2 && git checkout --detach {GIT_SHA}",
        "curl -fsSL https://deb.nodesource.com/setup_20.x | bash -",
        "apt-get install -y nodejs",
        "python -m pip install -U pip setuptools wheel packaging ninja",
        "python -m pip install 'numpy<2'",
        "python -m pip install -e /opt/testing2",
        "python -m pip install flash-attn --no-build-isolation",
        "cd /opt/testing2/metamon/env/vectorized && npm ci --omit=dev",
    )
)

app = modal.App("smallg1onlinev1a", image=image)
data_volume = modal.Volume.from_name(DATA_VOLUME_NAME, create_if_missing=False)
run_volume = modal.Volume.from_name(RUN_VOLUME_NAME, create_if_missing=False)
secret = modal.Secret.from_name(SECRET_NAME)


In [ ]:
POOL_YAML = r'''# SmallG1OnlineV1a pool: SmallG1OnlineV1.5 roster with V0 replaced by epoch 475.
defaults:
  team_set: hl_05_26
  battle_backend: metamon
  checkpoints: [null]
  temperatures: [0.5, 1.0, 1.25, 1.5, 1.75, 2.0]
  num_agents: 1
agents:
  TaurosV0:
    model_name: TaurosV0
    num_agents: 2
    checkpoints: "range(50, 63, 2)"
  Kakuna:
    model_name: Kakuna
    num_agents: 2
    checkpoints: "range(24, 35, 2)"
  Epoch475:
    model_name: TaurosV1B
    num_agents: 2
    checkpoints: [475]
  Alakazam:
    model_name: Alakazam
    checkpoints: "range(2, 9, 2)"
  SyntheticRLV2:
    model_name: SyntheticRLV2
    checkpoints: "range(36, 49, 2)"
  V2ADataAblation:
    model_name: V2AGroupedV2DataAblation
    checkpoints: "range(74, 91, 2)"
  Superkazam:
    model_name: Superkazam
    checkpoints: "range(38, 51, 2)"
  Kadabra3:
    model_name: Kadabra3
    checkpoints: "range(10, 21, 2)"
'''

RUN_YAML_TEMPLATE = r'''run_name: smallg1onlinev1a
base_model: TaurosV1A
battle_format: gen1ou
dataset_config: online_selfplay.yaml
train_gin_config: grouped_v2_large_isfilter.gin
train_pool: /data/config/hl_gen1ou_smallg1onlinev1a.yaml
train_team_set: smallg1onlinev1a_expert
val_pool: metamon/rl/configs/opponent_pools/taurosv1a_val.yaml
val_team_set: smallg1onlinev1a_expert
dset_max_size: 300000
dset_min_size: 5000
initial_online_weight: 0.0
online_weight: 0.4
online_anneal_start_epoch: 0
online_anneal_end_epoch: 20
learning_rate: 8.0e-5
lr_warmup_epochs: 20
seq_floor_warmup_epochs: 20
batch_size_per_gpu: 14
grad_accum: 1
epochs: 401
steps_per_epoch: 1000
ckpt_interval: 10
full_state_ckpt_interval: 10
mixed_precision: "no"
lanes: __LANES__
n_workers: 1
train_timesteps_per_epoch: 500
temp_low: 1.0
temp_high: 2.0
val_timesteps: 1000
val_interval: 10
dloader_workers: 6
'''

PUBLIC_OPPONENTS = {
    "TaurosV0": list(range(50, 63, 2)),
    "Kakuna": list(range(24, 35, 2)),
    "Alakazam": list(range(2, 9, 2)),
    "SyntheticRLV2": list(range(36, 49, 2)),
    "V2AGroupedV2DataAblation": list(range(74, 91, 2)),
    "Superkazam": list(range(38, 51, 2)),
    "Kadabra3": list(range(10, 21, 2)),
}


In [ ]:
@app.function(
    volumes={"/data": data_volume, "/run": run_volume},
    secrets=[secret],
    cpu=8,
    memory=32768,
    timeout=86400,
)
def bootstrap(weights_url: str = "", teams_url: str = ""):
    import hashlib
    import json
    import os
    import re
    import shutil
    import subprocess
    import tarfile
    import tempfile
    import zipfile
    from pathlib import Path

    import requests

    data_root = Path("/data")
    run_root = Path("/run")
    bootstrap_dir = data_root / "bootstrap"
    config_dir = data_root / "config"
    bootstrap_dir.mkdir(parents=True, exist_ok=True)
    config_dir.mkdir(parents=True, exist_ok=True)
    raw_weights = bootstrap_dir / "policy_epoch_475.pt"
    clean_weights = bootstrap_dir / "policy_epoch_475_online.pt"

    def download(url: str, destination: Path):
        headers = {}
        token = os.environ.get("GITHUB_TOKEN")
        if token:
            headers["Authorization"] = f"Bearer {token}"
        with requests.get(url, headers=headers, stream=True, timeout=120) as r:
            r.raise_for_status()
            with destination.open("wb") as f:
                for chunk in r.iter_content(8 * 1024 * 1024):
                    if chunk:
                        f.write(chunk)

    def release_asset(asset_names):
        headers = {"Accept": "application/vnd.github+json"}
        token = os.environ.get("GITHUB_TOKEN")
        if token:
            headers["Authorization"] = f"Bearer {token}"
        url = f"https://api.github.com/repos/{REPO}/releases/latest"
        r = requests.get(url, headers=headers, timeout=30)
        if r.status_code == 404:
            return None
        r.raise_for_status()
        by_name = {a["name"]: a["browser_download_url"] for a in r.json()["assets"]}
        return next((by_name[n] for n in asset_names if n in by_name), None)

    def sha256(path: Path) -> str:
        digest = hashlib.sha256()
        with path.open("rb") as f:
            for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
                digest.update(block)
        return digest.hexdigest()

    def assemble_parts(parts, checksum_path=None):
        if len(parts) != 3:
            raise ValueError(f"Expected exactly three epoch-475 shards, found {len(parts)}")
        for part in parts:
            if part.stat().st_size > 100 * 1024 * 1024:
                raise ValueError(f"Shard exceeds GitHub's 100 MiB hard limit: {part}")
        temporary = raw_weights.with_suffix(".pt.assembling")
        with temporary.open("wb") as output:
            for part in parts:
                with part.open("rb") as source:
                    shutil.copyfileobj(source, output, length=8 * 1024 * 1024)
        actual = sha256(temporary)
        if checksum_path is not None and checksum_path.exists():
            expected = checksum_path.read_text().split()[0].lower()
            if actual != expected:
                temporary.unlink(missing_ok=True)
                raise ValueError(f"Reassembled checkpoint SHA-256 mismatch: {actual} != {expected}")
        os.replace(temporary, raw_weights)
        print(f"Reassembled epoch 475: {raw_weights.stat().st_size:,} bytes, sha256={actual}")

    # Exact epoch 475 is part of V1a; do not silently substitute another epoch.
    # Prefer the three ordinary-Git shards from the cloned repository.
    repo_policy = Path("/opt/testing2/policy")
    repo_parts = sorted(repo_policy.glob("policy_epoch_475.pt.part-*"))
    repo_checksum = repo_policy / "policy_epoch_475.pt.sha256"
    if not raw_weights.exists() and repo_parts:
        print("Assembling epoch 475 from Git-tracked repository shards")
        assemble_parts(repo_parts, repo_checksum)

    if not raw_weights.exists():
        resolved = weights_url or release_asset(["policy_epoch_475.pt"])
        if resolved:
            print("Downloading monolithic epoch 475 from", resolved)
            download(resolved, raw_weights)
        else:
            part_urls = [
                release_asset([f"policy_epoch_475.pt.part-{i:03d}"])
                for i in range(3)
            ]
            if all(part_urls):
                with tempfile.TemporaryDirectory() as td:
                    parts = []
                    for i, url in enumerate(part_urls):
                        part = Path(td) / f"policy_epoch_475.pt.part-{i:03d}"
                        download(url, part)
                        parts.append(part)
                    checksum = Path(td) / "policy_epoch_475.pt.sha256"
                    checksum_url = release_asset(["policy_epoch_475.pt.sha256"])
                    if checksum_url:
                        download(checksum_url, checksum)
                    assemble_parts(parts, checksum if checksum.exists() else None)
            else:
                raise FileNotFoundError(
                    "No epoch-475 checkpoint found. Commit exactly three shards under policy/, "
                    "upload /data/bootstrap/policy_epoch_475.pt with `modal volume put`, set "
                    "WEIGHTS_URL, or publish matching GitHub Release assets."
                )

    # When a repository checksum exists, verify pre-uploaded and monolithic sources too.
    if repo_checksum.exists():
        expected = repo_checksum.read_text().split()[0].lower()
        actual = sha256(raw_weights)
        if actual != expected:
            raise ValueError(f"Epoch-475 checkpoint SHA-256 mismatch: {actual} != {expected}")

    source_teams = bootstrap_dir / "good_teams"
    if not source_teams.exists():
        resolved = teams_url or release_asset(["good_teams.tar.gz", "good_teams.tgz", "good_teams.zip"])
        if not resolved:
            raise FileNotFoundError(
                "Missing /data/bootstrap/good_teams. Upload the directory with `modal volume put` "
                "or publish a good_teams archive as a GitHub release asset."
            )
        with tempfile.TemporaryDirectory() as td:
            archive = Path(td) / Path(resolved.split("?")[0]).name
            download(resolved, archive)
            unpacked = Path(td) / "unpacked"
            unpacked.mkdir()
            if archive.suffix == ".zip":
                with zipfile.ZipFile(archive) as zf:
                    root = unpacked.resolve()
                    for member in zf.infolist():
                        if not (root / member.filename).resolve().is_relative_to(root):
                            raise ValueError(f"Unsafe archive member: {member.filename}")
                    zf.extractall(unpacked)
            else:
                with tarfile.open(archive, "r:*") as tf:
                    root = unpacked.resolve()
                    for member in tf.getmembers():
                        if not (root / member.name).resolve().is_relative_to(root):
                            raise ValueError(f"Unsafe archive member: {member.name}")
                    tf.extractall(unpacked)
            source_teams.mkdir(parents=True)
            archived_teams = sorted(
                p for p in unpacked.rglob("*")
                if p.is_file() and (p.name.endswith(".txt") or p.name.endswith(".gen1ou_team"))
            )
            for i, p in enumerate(archived_teams):
                shutil.copy2(p, source_teams / f"{i:04d}__{p.name}")

    subprocess.run([
        "python", "-m", "metamon.rl.taurosv1b_kl", "export-online",
        "--input", str(raw_weights), "--output", str(clean_weights),
    ], cwd="/opt/testing2", check=True)

    import torch
    state = torch.load(clean_weights, map_location="cpu")
    if any(k.startswith("_kl_anchor_") or k == "_kl_forward_step" for k in state):
        raise ValueError("KL-only keys remain after export-online")
    if len(state) != 598:
        raise ValueError(f"Expected 598 TaurosV1B online keys, found {len(state)}")
    n_params = sum(v.numel() for v in state.values())
    if n_params != 34_766_593:
        raise ValueError(f"Unexpected state size: {n_params:,} (expected 34,766,593)")

    # Flatten into unique ordinal names so duplicate team content and duplicate basenames survive.
    team_inputs = sorted(
        p for p in source_teams.rglob("*")
        if p.is_file() and (p.name.endswith(".txt") or p.name.endswith(".gen1ou_team"))
    )
    if len(team_inputs) != 807:
        raise ValueError(f"Expected exactly 807 source teams without deduplication, found {len(team_inputs)}")
    target_teams = data_root / "cache/teams/smallg1onlinev1a_expert/gen1ou"
    if target_teams.exists():
        shutil.rmtree(target_teams)
    target_teams.mkdir(parents=True)
    out_names = []
    for i, src in enumerate(team_inputs):
        stem = src.name.removesuffix(".gen1ou_team").removesuffix(".txt")
        safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", stem)
        name = f"{i:04d}__{safe}.gen1ou_team"
        shutil.copyfile(src, target_teams / name)
        out_names.append(name)

    # Sparse team exports need generation-correct defaults from usage stats while parsing.
    from metamon.data.download import download_usage_stats
    download_usage_stats(gen=1, version="v5", force_download=False)
    from metamon.backend.team_prediction.team import TeamSet as PredictionTeamSet
    for name in out_names:
        team = PredictionTeamSet.from_showdown_file(str(target_teams / name), "gen1ou")
        if len(team.pokemon) != 6:
            raise ValueError(f"{name}: parsed {len(team.pokemon)} Pokemon, expected 6")
    with (target_teams / "index.csv").open("w", encoding="utf-8") as f:
        f.write("filename\n")
        f.writelines(f"{name}\n" for name in out_names)

    # Install the same cleaned policy twice: frozen pool opponent and current trainee seed.
    frozen = run_root / "checkpoints/epoch475_frozen/ckpts/policy_weights/policy_epoch_475.pt"
    latest = run_root / "checkpoints/smallg1onlinev1a/ckpts/latest/policy.pt"
    frozen.parent.mkdir(parents=True, exist_ok=True)
    latest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(clean_weights, frozen)
    if not latest.exists():
        shutil.copy2(clean_weights, latest)

    (config_dir / "hl_gen1ou_smallg1onlinev1a.yaml").write_text(POOL_YAML)
    (config_dir / "smallg1onlinev1a.yaml").write_text(
        RUN_YAML_TEMPLATE.replace("__LANES__", str(LANES_PER_COLLECTOR))
    )

    # Required offline mixture and opponent team set. Replay stats are not used by this RL loader.
    from metamon.data.download import (
        download_parsed_replays, download_self_play_data, download_teams, download_usage_stats
    )
    download_parsed_replays("gen1ou", version="v6", force_download=False)
    for subset in ("pac-base", "pac-exploratory", "pac-tauros"):
        download_self_play_data(subset, "gen1ou", version="main", force_download=False, extract=False)
    download_teams("gen1ou", "hl_05_26", version="v5", force_download=False)
    download_usage_stats(gen=1, version="v5", force_download=False)

    # Prefetch every public historical checkpoint serially, avoiding first-use races.
    import metamon.rl.taurosv1b_online  # registers local TaurosV1B
    from metamon.rl.pretrained import get_pretrained_model
    for model_name, epochs in PUBLIC_OPPONENTS.items():
        model = get_pretrained_model(model_name)
        for epoch in epochs:
            path = model.get_path_to_checkpoint(epoch)
            if not Path(path).exists():
                raise FileNotFoundError(f"Failed to prefetch {model_name}@{epoch}: {path}")

    manifest = {
        "git_sha": GIT_SHA, "expert_teams": len(out_names),
        "policy_state_keys": len(state), "policy_state_elements": n_params,
        "raw_weights_bytes": raw_weights.stat().st_size,
        "raw_weights_sha256": sha256(raw_weights),
    }
    (bootstrap_dir / "manifest.json").write_text(json.dumps(manifest, indent=2))
    data_volume.commit()
    run_volume.commit()
    return manifest


In [ ]:
def _remote_helpers_source():
    # Kept as a no-op marker: the subprocess helpers live directly in the functions below.
    return None

@app.function(
    volumes={"/data": data_volume, "/run": run_volume},
    secrets=[secret],
    cpu=32,
    memory=65536,
    timeout=86400,
)
def prefill(collector_processes: int = 4, minimum_files: int = 5001):
    import os
    import subprocess
    import time
    from pathlib import Path

    run_config = "/data/config/smallg1onlinev1a.yaml"
    base_weights = "/data/bootstrap/policy_epoch_475_online.pt"
    buffer_dir = "/run/online_buffer"
    fifo = Path(buffer_dir) / "gen1ou"
    fifo.mkdir(parents=True, exist_ok=True)

    def count_files():
        return sum(1 for p in fifo.rglob("*") if p.is_file() and p.name.endswith((".json", ".json.lz4")))

    current = count_files()
    if current >= minimum_files:
        return {"fifo_files": current, "already_ready": True}

    procs = []
    for seed in range(collector_processes):
        env = os.environ.copy()
        env["CUDA_VISIBLE_DEVICES"] = ""
        env["METAMON_CPU_COLLECTOR"] = "1"
        cmd = [
            "python", "-m", "metamon.rl.taurosv1b_online",
            "--run_config", run_config, "--mode", "collect",
            "--save_dir", "/run/checkpoints", "--buffer_dir", buffer_dir,
            "--base_weights", base_weights, "--epochs", "1000000",
            "--seed", str(seed),
        ]
        procs.append(subprocess.Popen(cmd, cwd="/opt/testing2", env=env))
    try:
        while True:
            failed = [(i, p.returncode) for i, p in enumerate(procs) if p.poll() not in (None, 0)]
            if failed:
                raise RuntimeError(f"Collector failure(s): {failed}")
            current = count_files()
            print(f"FIFO prefill: {current:,}/{minimum_files:,}", flush=True)
            if current >= minimum_files:
                break
            time.sleep(30)
    finally:
        for p in procs:
            if p.poll() is None:
                p.terminate()
        for p in procs:
            try:
                p.wait(timeout=30)
            except subprocess.TimeoutExpired:
                p.kill()
    run_volume.commit()
    return {"fifo_files": count_files(), "already_ready": False}


In [ ]:
@app.function(
    volumes={"/data": data_volume, "/run": run_volume},
    secrets=[secret],
    gpu=GPU,
    cpu=32,
    memory=98304,
    timeout=86400,
    retries=3,
)
def train(collector_processes: int = 4):
    import os
    import re
    import shutil
    import subprocess
    from pathlib import Path

    run_name = "smallg1onlinev1a"
    run_config = "/data/config/smallg1onlinev1a.yaml"
    base_weights = "/data/bootstrap/policy_epoch_475_online.pt"
    buffer_dir = "/run/online_buffer"
    ckpt = Path("/run/checkpoints") / run_name / "ckpts"
    state_root = ckpt / "training_states"
    state_epochs = []
    if state_root.exists():
        for p in state_root.iterdir():
            m = re.fullmatch(rf"{re.escape(run_name)}_epoch_(\d+)", p.name)
            if p.is_dir() and m:
                state_epochs.append(int(m.group(1)))

    resume_epoch = max(state_epochs) if state_epochs else None
    latest = ckpt / "latest/policy.pt"
    latest.parent.mkdir(parents=True, exist_ok=True)
    if resume_epoch is None:
        shutil.copy2(base_weights, latest)
    else:
        matching_policy = ckpt / f"policy_weights/policy_epoch_{resume_epoch}.pt"
        if not matching_policy.exists():
            raise FileNotFoundError(f"Full state exists without matching policy: {matching_policy}")
        # Roll latest back to the exact full-state epoch before collectors start.
        shutil.copy2(matching_policy, latest)

    collector_procs = []
    for seed in range(collector_processes):
        env = os.environ.copy()
        env["CUDA_VISIBLE_DEVICES"] = ""
        env["METAMON_CPU_COLLECTOR"] = "1"
        cmd = [
            "python", "-m", "metamon.rl.taurosv1b_online",
            "--run_config", run_config, "--mode", "collect",
            "--save_dir", "/run/checkpoints", "--buffer_dir", buffer_dir,
            "--base_weights", str(latest), "--epochs", "1000000",
            "--seed", str(1000 + seed),
        ]
        collector_procs.append(subprocess.Popen(cmd, cwd="/opt/testing2", env=env))

    learner = [
        "python", "-m", "metamon.rl.taurosv1b_online",
        "--run_config", run_config, "--mode", "learn",
        "--save_dir", "/run/checkpoints", "--buffer_dir", buffer_dir, "--log",
    ]
    if resume_epoch is None:
        learner += ["--base_weights", base_weights]
    else:
        learner += ["--resume_training_state", "--resume_epoch", str(resume_epoch)]

    try:
        print("Launching learner; resume_epoch=", resume_epoch, flush=True)
        completed = subprocess.run(learner, cwd="/opt/testing2")
        if completed.returncode != 0:
            raise subprocess.CalledProcessError(completed.returncode, learner)
    finally:
        for p in collector_procs:
            if p.poll() is None:
                p.terminate()
        for p in collector_procs:
            try:
                p.wait(timeout=30)
            except subprocess.TimeoutExpired:
                p.kill()
        run_volume.commit()
    return {"completed": True, "resumed_from": resume_epoch}

@app.function(volumes={"/data": data_volume, "/run": run_volume}, timeout=1800)
def status():
    import json
    import re
    import subprocess
    from pathlib import Path

    fifo = Path("/run/online_buffer/gen1ou")
    fifo_count = sum(1 for p in fifo.rglob("*") if p.is_file() and p.name.endswith((".json", ".json.lz4"))) if fifo.exists() else 0
    policies = Path("/run/checkpoints/smallg1onlinev1a/ckpts/policy_weights")
    epochs = []
    if policies.exists():
        for p in policies.glob("policy_epoch_*.pt"):
            m = re.search(r"(\d+)$", p.stem)
            if m:
                epochs.append(int(m.group(1)))
    data_du = subprocess.run(["du", "-sh", "/data"], text=True, capture_output=True).stdout.strip()
    run_du = subprocess.run(["du", "-sh", "/run"], text=True, capture_output=True).stdout.strip()
    manifest_path = Path("/data/bootstrap/manifest.json")
    return {
        "fifo_files": fifo_count,
        "latest_policy_epoch": max(epochs) if epochs else None,
        "data_volume_du": data_du,
        "run_volume_du": run_du,
        "manifest": json.loads(manifest_path.read_text()) if manifest_path.exists() else None,
    }


## 1. Bootstrap data and verify the proposal

The first run builds the CUDA image, downloads several large datasets, validates every team, strips TaurosV1B's KL-anchor-only tensors, checks the resulting 598-key / 34,766,593-element policy state, and prefetches all public opponents. It is idempotent.

In [ ]:
with modal.enable_output():
    with app.run():
        bootstrap_result = bootstrap.remote(WEIGHTS_URL, TEAMS_URL)
bootstrap_result


## 2. CPU-only FIFO prefill

This blocks until the audited loader's readiness condition is satisfied (`> 5000`, hence 5,001 files). Re-running is safe and returns immediately once ready.

In [ ]:
with modal.enable_output():
    with app.run():
        prefill_result = prefill.remote(COLLECTOR_PROCESSES, 5001)
prefill_result


## 3. Launch training asynchronously

The call is durable after the notebook cell exits. It uses a fixed W&B run ID with resume enabled. Full model/optimizer/scheduler/RNG state is saved every 10 epochs. Modal permits at most 24 hours per function attempt, so three retries are configured; each attempt finds the newest complete full state, restores its matching policy as `latest`, and resumes at the next epoch. If all attempts are consumed before epoch 400, re-run this cell; it resumes rather than restarts.

In [ ]:
with modal.enable_output():
    with app.run():
        training_call = train.spawn(COLLECTOR_PROCESSES)
        training_call_id = training_call.object_id
print("Training call ID:", training_call_id)
print("W&B project: smallg1onlinev1a; run ID: smallg1onlinev1a-modal-v1")


## Inspect status or retrieve checkpoints

The status function is read-only. To download all V1a checkpoints later:

```bash
modal volume get smallg1onlinev1a-run /checkpoints/smallg1onlinev1a ./smallg1onlinev1a
```

In [ ]:
with app.run():
    current_status = status.remote()
current_status
